# Title

Angos and Tan

BSDSBA 2028

In [19]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

### Data Download

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("usdot/flight-delays", output_dir="./data")

# print("Path to dataset files:", path)

# OVERVIEW

This mini-project analyzes airline operations using the **U.S. Department of Transportation (DOT) 2015 Flight Delays and Cancellations dataset**. The project investigates how airport congestion, airline scheduling practices, and traffic density contribute to flight delays. In addition to identifying structural causes of delays, the project aims to forecast flight delay duration based on operational conditions, including airport traffic, route characteristics, and schedule realism.

Airline delays are often attributed to weather or operational disruptions, but underlying causes may include unrealistic scheduling and airport capacity constraints. By analyzing patterns in flight schedules, traffic density, and delay distributions, this project aims to identify systemic patterns that contribute to delays and congestion.

Understanding these factors has practical applications for airlines, airports, and passengers. Airlines can improve schedule planning, airports can identify congestion thresholds, and travelers can better understand which routes or airports are more reliable.


## GOALS

1. Analyze patterns in flight delays across airlines, routes, airports, and time of day.

2. Identify congestion thresholds at which airport traffic density results in significant increases in delay.

3. Forecast expected delay duration for flights using operational features such as airport traffic, route reliability, and schedule realism.

4. Evaluate whether airline schedules are realistic by comparing scheduled flight time with actual flight durations.


## Basic

### Data Loading

In [3]:
# Load Data
airlines = pd.read_csv("data/airlines.csv")
airports = pd.read_csv("data/airports.csv")
flights = pd.read_csv("data/flights.csv", low_memory=False)

In [4]:
display(flights.head())
print(flights.info())
print(flights.shape)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              str    
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          str    
 7   ORIGIN_AIRPORT       str    
 8   DESTINATION_AIRPORT  str    
 9   SCHEDULED_DEPARTURE  int64  
 10  DEPARTURE_TIME       float64
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           float64
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            float64
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    int64  
 21  ARRIVAL_TIME         float64
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24  CANCELLED

### DATA CLEANING

#### Fix 5-digit airport codes to standard 3-character IATA codes

Some data points use 5-digit airport codes instead of the standard IATA codes:

In [6]:
flights[flights['ORIGIN_AIRPORT'].str.len()!=3].head()[['ORIGIN_AIRPORT','DESTINATION_AIRPORT']]

,ORIGIN_AIRPORT,DESTINATION_AIRPORT
4385712,14747,11298
4385713,14771,13487
4385714,12889,13487
4385715,12892,13303
4385716,14771,11057


Also, comparing airports present in our `flights` and `airports` dataframe, one airport is missing. To solve this, that airport will be removed from the list of flights entirely. 

We'll utilize a helper function to fix these two issues.

In [ ]:
import clean_flights

clean_flights.clean('data/flights.csv')

### FEATURE ENGINEERING

The cleaned flights will be used instead of the original csv file.

In [11]:
flights = pd.read_csv("data/flights_filtered_fixed.csv", low_memory=False)

### Airport-level features:

- `avg_flights_per_hour`: $\frac{\text{total flights for airport}}{\text{number of hours observed}}$
- `avg_delay_rate`: P(delay)

In [12]:
filtered_flights = flights[flights['CANCELLED'] == 0]
valid_flights = flights[
    (flights['CANCELLED'] == 0) &
    (flights['DIVERTED'] == 0) &
    (flights['MONTH'] != 10)
]


airports['avg_flights_per_hour'] = airports['IATA_CODE'].map(
    flights.assign(hour=filtered_flights['SCHEDULED_DEPARTURE']//100)
           .groupby(['ORIGIN_AIRPORT','hour'])
           .size()
           .groupby('ORIGIN_AIRPORT')
           .mean()
           .round()
)

airports['avg_flights_per_hour'] = airports['IATA_CODE'].map(
    flights.assign(hour=filtered_flights['SCHEDULED_DEPARTURE']//100)
           .groupby(['ORIGIN_AIRPORT','hour'])
           .size()
           .groupby('ORIGIN_AIRPORT')
           .mean()
           .round()
)

airport_delay = (
    valid_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby('ORIGIN_AIRPORT')['delayed']
    .mean()
)

airports['AVG_DELAY_RATE'] = airports['IATA_CODE'].map(airport_delay)

display(airports.head())


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,avg_flights_per_hour,AVG_DELAY_RATE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040,247.0,0.370903
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190,173.0,0.307485
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,1039.0,0.368168
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,73.0,0.371041
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,95.0,0.356813


### Route-level features

- `schedule_padding` - $\text{average scheduled time} - \text{median flight time}$

- `ave_delay` - $\frac{\sum \text{arrival delays}}{\text{num flights for the route}}$

- `delay-rate` - P(delay)

In [13]:
route_df = (
    valid_flights
    .groupby(['ORIGIN_AIRPORT','DESTINATION_AIRPORT'], as_index=False)
    .agg(
        NUM_FLIGHTS=('ELAPSED_TIME','size'),
        AVE_FLIGHT_TIME=('ELAPSED_TIME','mean'),
        MEDIAN_FLIGHT_TIME=('ELAPSED_TIME','median'),
        AVE_SCHEDULED_TIME=('SCHEDULED_TIME','mean')
    )
)

route_df['SCHEDULE_PADDING'] = (
    route_df['AVE_SCHEDULED_TIME'] -
    route_df['MEDIAN_FLIGHT_TIME']
)

route_df['AVE_DELAY'] = (
    valid_flights
    .groupby(['ORIGIN_AIRPORT','DESTINATION_AIRPORT'])['ARRIVAL_DELAY']
    .mean()
    .values
)

delay_prob = (
    valid_flights
    .assign(delayed=lambda x: x['ARRIVAL_DELAY'] > 0)
    .groupby(['ORIGIN_AIRPORT','DESTINATION_AIRPORT'])['delayed']
    .mean()
)

route_df['DELAY_RATE'] = delay_prob.values

display(route_df.head())

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,NUM_FLIGHTS,AVE_FLIGHT_TIME,MEDIAN_FLIGHT_TIME,AVE_SCHEDULED_TIME,SCHEDULE_PADDING,AVE_DELAY,DELAY_RATE
0,ABE,ATL,886,127.415350,125.0,130.916479,5.916479,2.256208,0.336343
1,ABE,DTW,695,101.923741,100.0,104.417266,4.417266,7.925180,0.385612
2,ABE,ORD,646,130.298762,127.5,131.216718,3.716718,9.924149,0.402477
3,ABI,DFW,2231,53.951591,51.0,56.427163,5.427163,3.272075,0.307485
4,ABQ,ATL,799,174.822278,174.0,182.504380,8.504380,-0.093867,0.215269


## Airport reliability:

In [ ]:
display(airports.sort_values(by='AVG_DELAY_RATE', ascending=False).head(10).reset_index(drop=True)[['AIRPORT','AVG_DELAY_RATE']])

,AIRPORT,AVG_DELAY_RATE
0,Gustavus Airport,0.671053
1,Adak Airport,0.636364
2,Pago Pago International Airport (Tafuna Airport),0.632075
3,Wilmington Airport,0.557895
4,King Salmon Airport,0.507937
5,St. Cloud Regional Airport,0.493506
6,Rhinelander-Oneida County Airport,0.490928
7,Nome Airport,0.490506
8,Yellowstone Regional Airport,0.480799
9,Plattsburgh International Airport,0.467626


In [35]:
px.scatter(
    airports,
    x = 'avg_flights_per_hour',
    y = 'AVG_DELAY_RATE',
    labels = {'avg_flights_per_hour': 'Average Flights per Hour',
              'AVG_DELAY_RATE': 'Average Delay Rate'},
    title = 'Airport-level Traffic to delay rate relationship',
    subtitle= 'There is a slight positive trend showing that delay rates increase as airports faces higher hourly flights.',
    trendline='ols'
)

In [53]:
airport_hour_df = (
    valid_flights
    .assign(
        hour = valid_flights['SCHEDULED_DEPARTURE'] // 100,
        delayed = valid_flights['ARRIVAL_DELAY'] > 0
    )
    .groupby(['ORIGIN_AIRPORT','hour'])
    .agg(
        flights=('FLIGHT_NUMBER','size'),
        avg_delay=('ARRIVAL_DELAY', lambda x: x[x > 0].mean()),
        delay_rate=('delayed','mean')
    )
    .reset_index()
)

In [54]:
airport_hour_df.sort_values(by='delay_rate')

,ORIGIN_AIRPORT,hour,flights,avg_delay,delay_rate
55,ABY,7,2,NaN,0.0
52,ABR,17,3,NaN,0.0
1454,FCA,19,1,NaN,0.0
541,BRD,11,1,NaN,0.0
1964,IDA,9,1,NaN,0.0
...,...,...,...,...,...
3272,PSC,18,4,64.75,1.0
3320,PVD,22,1,26.00,1.0
3422,RKS,8,2,198.50,1.0
3428,RKS,18,1,18.00,1.0


In [55]:
fig = px.scatter(
    airport_hour_df,
    x='hour',
    y='delay_rate',
    trendline='ols',
    opacity=0.4,
    hover_data = {'ORIGIN_AIRPORT': True}
)

fig.show()

## Intermediate

## Advanced